# Считаю retrieval метрики по DINOv2 

In [1]:
import polars as pl
import pandas as pd
import numpy as np
from pathlib import Path
import chromadb
import os
from dotenv import load_dotenv
from sneakers_hse.data.utils.s3_tools import S3Client
from sklearn.model_selection import train_test_split

load_dotenv()

False

In [ ]:
os.environ['AWS_ACCESS_KEY'] = ""
os.environ['AWS_SECRET_KEY'] = ""
os.environ['PROJECT_ROOT_PATH'] = ""

In [3]:
# PROJECT_ROOT_PATH = Path('/Users/alexansemyachkin/Desktop/studies/hse/sneakers-hse')
PROJECT_ROOT_PATH = Path(os.getenv('PROJECT_ROOT_PATH'))
# Скачиваю эмбеддинги из s3
s3 = S3Client(aws_access_key_id=os.getenv("AWS_ACCESS_KEY"),
              aws_secret_access_key=os.getenv("AWS_SECRET_KEY"))

In [19]:
s3._download_one(
    bucket_name='sneakers-hse-images-test',
    s3_key='dinov2/embeddings.parquet.gzip', # Путь пишем с учётом модели, т.е. каждой модели будет соответстовать отдельный инстанс ChromaDB
    local_path=str(PROJECT_ROOT_PATH / 'data/04_dinov2_embeddings.parquet.gzip')
)

'/Users/a.akhmadullin/vscode-projects/sneakers-hse/data/04_dinov2_embeddings.parquet.gzip'

In [25]:
s3._download_one(
    bucket_name='sneakers-hse-images-test',
    s3_key='train_images.csv', # Путь пишем с учётом модели, т.е. каждой модели будет соответстовать отдельный инстанс ChromaDB
    local_path=str(PROJECT_ROOT_PATH / 'data/03_yolo_preprocessed/train_images.csv')
)

'/Users/a.akhmadullin/vscode-projects/sneakers-hse/data/03_yolo_preprocessed/train_images.csv'

In [26]:
s3._download_one(
    bucket_name='sneakers-hse-images-test',
    s3_key='test_images.csv', # Путь пишем с учётом модели, т.е. каждой модели будет соответстовать отдельный инстанс ChromaDB
    local_path=str(PROJECT_ROOT_PATH / 'data/03_yolo_preprocessed/test_images.csv')
)

'/Users/a.akhmadullin/vscode-projects/sneakers-hse/data/03_yolo_preprocessed/test_images.csv'

In [4]:
s3.download_folder_from_s3_parallel(
    bucket_name='sneakers-hse-images-test',
    s3_prefix='dinov2/chroma_db', # Путь пишем с учётом модели, т.е. каждой модели будет соответстовать отдельный инстанс ChromaDB
    local_folder=str(PROJECT_ROOT_PATH / 'chroma_db'),
    max_workers=10
)

Total files: 27
Downloaded: /Users/a.akhmadullin/vscode-projects/sneakers-hse/chroma_db/76efa518-b14e-4991-b2c5-3211233aa8ef/link_lists.bin
Downloaded: /Users/a.akhmadullin/vscode-projects/sneakers-hse/chroma_db/76efa518-b14e-4991-b2c5-3211233aa8ef/length.bin
Downloaded: /Users/a.akhmadullin/vscode-projects/sneakers-hse/chroma_db/a44b49bf-bdb0-4ddd-be29-5418ddff8a86/header.bin
Downloaded: /Users/a.akhmadullin/vscode-projects/sneakers-hse/chroma_db/1a6a579a-10d8-4d7c-a0ac-cbe18ec0ee40/length.bin
Downloaded: /Users/a.akhmadullin/vscode-projects/sneakers-hse/chroma_db/76efa518-b14e-4991-b2c5-3211233aa8ef/header.bin
Downloaded: /Users/a.akhmadullin/vscode-projects/sneakers-hse/chroma_db/1a6a579a-10d8-4d7c-a0ac-cbe18ec0ee40/link_lists.bin
Downloaded: /Users/a.akhmadullin/vscode-projects/sneakers-hse/chroma_db/a44b49bf-bdb0-4ddd-be29-5418ddff8a86/link_lists.bin
Downloaded: /Users/a.akhmadullin/vscode-projects/sneakers-hse/chroma_db/a44b49bf-bdb0-4ddd-be29-5418ddff8a86/length.bin
Downloaded: 

In [4]:
df = pl.read_parquet(PROJECT_ROOT_PATH / "data/04_dinov2_embeddings.parquet.gzip")

embeddings = np.stack(df["embedding"].to_list()).astype("float32")
labels = np.array(df["class"].to_list())
paths = df["path"].to_list()
df['class'].value_counts()

class,count
str,u32
"""Nike Кеды Dunk Low Retro""",287
"""Reebok Кроссовки CLASSIC NYLON""",286
"""Vans Кеды Upland""",275
"""PUMA Кроссовки Hypnotic LS""",282
"""Kari Кеды""",273
…,…
"""Reebok Кеды CLUB C 85""",283
"""adidas Кроссовки RUNFALCON 5""",285
"""Maison David Кроссовки""",286


In [5]:
df[0].select(pl.col('embedding')).to_numpy()[0][0].__len__()

768

In [6]:
train_df_pre = pl.read_csv(PROJECT_ROOT_PATH / 'data/03_yolo_preprocessed/train_images.csv')
test_df = pl.read_csv(PROJECT_ROOT_PATH / 'data/03_yolo_preprocessed/test_images.csv')

train_df, val_df = train_test_split(
    train_df_pre,
    test_size=0.2,
    stratify=train_df_pre["sneaker_class"],
    random_state=42
)

display(train_df.head(), train_df.shape)
display(val_df.head(), val_df.shape)
display(test_df.head(), test_df.shape)

path,sneaker_class
str,str
"""converse_chuck_taylor_all-star…","""converse_chuck_taylor_all-star…"
"""nike_air_max_270/0069.jpg""","""nike_air_max_270"""
"""converse_chuck_70_high/0008.jp…","""converse_chuck_70_high"""
"""adidas_stan_smith/0086.jpg""","""adidas_stan_smith"""
"""nike_air_jordan_3/0069.jpg""","""nike_air_jordan_3"""


(3708, 2)

path,sneaker_class
str,str
"""nike_air_max_plus_(tn)/0054.jp…","""nike_air_max_plus_(tn)"""
"""adidas_forum_high/0075.jpg""","""adidas_forum_high"""
"""adidas_gazelle/0110.jpg""","""adidas_gazelle"""
"""new_balance_574/0120.jpg""","""new_balance_574"""
"""reebok_club_c_85/0074.jpg""","""reebok_club_c_85"""


(928, 2)

path,sneaker_class
str,str
"""adidas_forum_low/0026.jpg""","""adidas_forum_low"""
"""nike_air_jordan_4/0078.jpg""","""nike_air_jordan_4"""
"""yeezy_boost_350_v2/0018.jpg""","""yeezy_boost_350_v2"""
"""adidas_superstar/0028.jpg""","""adidas_superstar"""
"""adidas_forum_high/0091.jpg""","""adidas_forum_high"""


(1160, 2)

In [7]:
df

path,class,embedding
str,str,list[f64]
"""Vans Кеды Upland/clothing_0_26…","""Vans Кеды Upland""","[-1.930393, -0.157552, … 0.300103]"
"""Vans Кеды Upland/clothing_0_57…","""Vans Кеды Upland""","[-1.749115, -1.480964, … -0.966375]"
"""Vans Кеды Upland/orig_45.jpeg""","""Vans Кеды Upland""","[0.503809, -0.948843, … -3.250608]"
"""Vans Кеды Upland/clothing_0_0.…","""Vans Кеды Upland""","[-1.441995, -2.107342, … -0.808429]"
"""Vans Кеды Upland/clothing_0_23…","""Vans Кеды Upland""","[-0.845633, -2.200308, … -1.242736]"
…,…,…
"""Under Armour Кроссовки UA Phan…","""Under Armour Кроссовки UA Phan…","[0.131167, -0.318168, … -0.66955]"
"""Under Armour Кроссовки UA Phan…","""Under Armour Кроссовки UA Phan…","[1.267569, -3.275375, … -2.853185]"
"""Under Armour Кроссовки UA Phan…","""Under Armour Кроссовки UA Phan…","[1.495174, -1.613851, … -1.374945]"


In [8]:
sql = pl.SQLContext()
sql.register_globals()
df = sql.execute(
    '''
select
    df.*
    , case when train_df.path is not null then 'train'
            when val_df.path is not null then 'val'
            when test_df.path is not null then 'test'
            end as sample_part
from df
left join train_df using(path)
left join val_df using(path)
left join test_df using(path)
'''
).collect()
df

path,class,embedding,sample_part
str,str,list[f64],str
"""Vans Кеды Upland/clothing_0_26…","""Vans Кеды Upland""","[-1.930393, -0.157552, … 0.300103]",null
"""Vans Кеды Upland/clothing_0_57…","""Vans Кеды Upland""","[-1.749115, -1.480964, … -0.966375]",null
"""Vans Кеды Upland/orig_45.jpeg""","""Vans Кеды Upland""","[0.503809, -0.948843, … -3.250608]",null
"""Vans Кеды Upland/clothing_0_0.…","""Vans Кеды Upland""","[-1.441995, -2.107342, … -0.808429]",null
"""Vans Кеды Upland/clothing_0_23…","""Vans Кеды Upland""","[-0.845633, -2.200308, … -1.242736]",null
…,…,…,…
"""Under Armour Кроссовки UA Phan…","""Under Armour Кроссовки UA Phan…","[0.131167, -0.318168, … -0.66955]",null
"""Under Armour Кроссовки UA Phan…","""Under Armour Кроссовки UA Phan…","[1.267569, -3.275375, … -2.853185]",null
"""Under Armour Кроссовки UA Phan…","""Under Armour Кроссовки UA Phan…","[1.495174, -1.613851, … -1.374945]",null


In [9]:
client = chromadb.Client()
collection = client.get_or_create_collection(
    "embeddings",
    metadata={"hnsw:space": "cosine"})

def add_in_batches(collection, ids, embeddings, metadatas, batch_size=5000):
    for i in range(0, len(ids), batch_size):
        collection.add(
            ids=ids[i:i+batch_size],
            embeddings=embeddings[i:i+batch_size],
            metadatas=metadatas[i:i+batch_size]
        )

add_in_batches(
    collection,
    ids=paths,
    embeddings=embeddings,
    metadatas=[{"class": c} for c in labels]
)

In [10]:
from tqdm.notebook import tqdm

def get_neighbors(collection, embeddings, k=10, batch_size=10):
    all_results = []
    for i in tqdm(range(0, len(embeddings), batch_size), desc="Querying"):
        batch = embeddings[i:i+batch_size]
        results = collection.query(
            query_embeddings=batch,
            n_results=k + 1  # +1 чтобы убрать self
        )['metadatas']
        all_results.extend(results)
    return np.array([[neighbor['class'] for neighbor in query]
                     for query in all_results])[:, 1:]  # Убираю self-match

def hit_rate_at_k(neighbors, labels, k):
    hits = 0
    for i in range(len(labels)):
        query_label = labels[i]
        retrieved = neighbors[i][:k]
        if query_label in retrieved:
            hits += 1
    return hits / len(labels)

def precision_at_k(neighbors, labels, k):
    precisions = []
    for i in range(len(labels)):
        query_label = labels[i]
        retrieved = neighbors[i][:k]
        relevant = (retrieved == query_label).sum()
        precisions.append(relevant / k)
    return np.mean(precisions)

def average_precision(retrieved, query_label, k):
    score = 0.0
    hits = 0
    for i in range(k):
        if retrieved[i] == query_label:
            hits += 1
            score += hits / (i + 1)
    return score / hits if hits > 0 else 0


def map_at_k(neighbors, labels, k):
    scores = []
    for i in range(len(labels)):
        query_label = labels[i]
        retrieved = neighbors[i][:k]
        scores.append(average_precision(retrieved, query_label, k))
    return np.mean(scores)

neighbors = get_neighbors(collection, embeddings, k=20)
len(neighbors[0])
for k in [1, 5, 10]:
    print(f"\nMetrics @ {k}")
    print("HitRate:", hit_rate_at_k(neighbors, labels, k))
    print("Precision:", precision_at_k(neighbors, labels, k))
    print("MAP:", map_at_k(neighbors, labels, k))

Querying:   0%|          | 0/1127 [00:00<?, ?it/s]


Metrics @ 1
HitRate: 0.7298712827341323
Precision: 0.7298712827341323
MAP: 0.7298712827341323

Metrics @ 5
HitRate: 0.8917887261429206
Precision: 0.6077407900577009
MAP: 0.7621327119396362

Metrics @ 10
HitRate: 0.934664891256103
Precision: 0.5363692853972482
MAP: 0.7185874593792079


In [11]:
df = df.with_columns(neighbors=neighbors)
df = df.with_columns(
    pl.struct(["neighbors", "class"]).map_elements(
        lambda row: [
            1 if n == row["class"] else 0
            for n in row["neighbors"][:k]
        ]
    ).alias("hit_flg")
)
df

path,class,embedding,sample_part,neighbors,hit_flg
str,str,list[f64],str,"array[str, 20]",list[i64]
"""Vans Кеды Upland/clothing_0_26…","""Vans Кеды Upland""","[-1.930393, -0.157552, … 0.300103]",null,"[""Vans Кеды Hylane"", ""PUMA Кроссовки RS-X Efekt PRM"", … ""PUMA Кроссовки Darter Pro""]","[0, 0, … 0]"
"""Vans Кеды Upland/clothing_0_57…","""Vans Кеды Upland""","[-1.749115, -1.480964, … -0.966375]",null,"[""Vans Кеды Upland"", ""Vans Кеды Upland"", … ""Vans Кеды Upland""]","[1, 1, … 0]"
"""Vans Кеды Upland/orig_45.jpeg""","""Vans Кеды Upland""","[0.503809, -0.948843, … -3.250608]",null,"[""Nike Кеды NIKE AIR FORCE 1"", ""PUMA Кеды CA Pro Classic II"", … ""Nike Кеды NIKE AIR FORCE 1""]","[0, 0, … 1]"
"""Vans Кеды Upland/clothing_0_0.…","""Vans Кеды Upland""","[-1.441995, -2.107342, … -0.808429]",null,"[""Vans Кеды Upland"", ""Vans Кеды Upland"", … ""PUMA Кеды Palermo""]","[1, 1, … 0]"
"""Vans Кеды Upland/clothing_0_23…","""Vans Кеды Upland""","[-0.845633, -2.200308, … -1.242736]",null,"[""Vans Кеды Hylane"", ""Vans Кеды Upland"", … ""Vans Кеды Hylane""]","[0, 1, … 1]"
…,…,…,…,…,…
"""Under Armour Кроссовки UA Phan…","""Under Armour Кроссовки UA Phan…","[0.131167, -0.318168, … -0.66955]",null,"[""Under Armour Кроссовки UA Phantom 4"", ""Under Armour Кроссовки UA Phantom 4"", … ""Under Armour Кроссовки UA Charged Rogue 5""]","[1, 1, … 0]"
"""Under Armour Кроссовки UA Phan…","""Under Armour Кроссовки UA Phan…","[1.267569, -3.275375, … -2.853185]",null,"[""Under Armour Кроссовки UA Phantom 4"", ""Under Armour Кроссовки UA Phantom 4"", … ""Under Armour Кроссовки UA Phantom 4""]","[1, 1, … 1]"
"""Under Armour Кроссовки UA Phan…","""Under Armour Кроссовки UA Phan…","[1.495174, -1.613851, … -1.374945]",null,"[""Under Armour Кроссовки UA Phantom 4"", ""Under Armour Кроссовки UA Phantom 4"", … ""Under Armour Кроссовки UA Phantom 4""]","[1, 1, … 1]"


In [12]:
for k in [1, 5, 10]:
    print(f"\nMetrics @ {k}")
    df_k = df.with_columns(hits_k = pl.col("hit_flg").list.slice(0, k))\
        .with_columns(
            pos = pl.int_ranges(1, pl.col("hits_k").list.len() + 1),
            hit_at_k = pl.col("hits_k").list.max(),
            precision_at_k = pl.col("hits_k").list.mean()
        )
    display(df_k.select(
        pl.col('hit_at_k').mean(),
        pl.col('precision_at_k').mean()
    ))
    display(df_k.group_by('class').agg(
        pl.col('hit_at_k').mean(),
        pl.col('precision_at_k').mean()
    ).sort(pl.col('hit_at_k'), descending=True))


Metrics @ 1


hit_at_k,precision_at_k
f64,f64
0.729871,0.729871


class,hit_at_k,precision_at_k
str,f64,f64
"""PUMA Кроссовки Darter Pro""",0.910035,0.910035
"""Nike Кроссовки AIR MAX 90""",0.902439,0.902439
"""Nike Кроссовки Nike Zoom Vomer…",0.893617,0.893617
"""Nike Кеды Dunk Low Retro""",0.885017,0.885017
"""Nike Кроссовки AIR PEGASUS 200…",0.881295,0.881295
…,…,…
"""Tendance Кроссовки""",0.505535,0.505535
"""adidas Кроссовки DURAMO SL2 M""",0.493952,0.493952
"""X-Plode Кроссовки""",0.446097,0.446097



Metrics @ 5


hit_at_k,precision_at_k
f64,f64
0.891789,0.607741


class,hit_at_k,precision_at_k
str,f64,f64
"""Nike Кроссовки Nike Zoom Vomer…",0.971631,0.839007
"""Nike Кеды Dunk Low Retro""",0.965157,0.777003
"""Nike Кроссовки AIR MAX 90""",0.954704,0.83554
"""adidas Кроссовки RUNFALCON 5""",0.954386,0.619649
"""PUMA Кроссовки Puma Morphic""",0.954198,0.803817
…,…,…
"""Under Armour Кроссовки UA CHAR…",0.783394,0.518412
"""Tendance Кроссовки""",0.723247,0.339483
"""X-Plode Кроссовки""",0.713755,0.326394



Metrics @ 10


hit_at_k,precision_at_k
f64,f64
0.934665,0.536369


class,hit_at_k,precision_at_k
str,f64,f64
"""adidas Кроссовки RUNFALCON 5""",0.982456,0.545263
"""Nike Кроссовки Nike Zoom Vomer…",0.98227,0.786879
"""Reebok Кроссовки RBK PREMIER T…",0.973585,0.546038
"""Nike Кеды Dunk Low Retro""",0.972125,0.699303
"""Under Armour Кроссовки UA Char…",0.971831,0.502113
…,…,…
"""Under Armour Кроссовки UA CHAR…",0.859206,0.470397
"""X-Plode Кроссовки""",0.821561,0.280297
"""Tendance Кроссовки""",0.808118,0.278598


In [13]:
sample_part = 'train'
for k in [1, 5, 10]:
    print(f"\nMetrics @ {k}")
    df_k = df.filter(pl.col('sample_part') == sample_part).with_columns(hits_k = pl.col("hit_flg").list.slice(0, k))\
        .with_columns(
            pos = pl.int_ranges(1, pl.col("hits_k").list.len() + 1),
            hit_at_k = pl.col("hits_k").list.max(),
            precision_at_k = pl.col("hits_k").list.mean()
        )
    display(df_k.select(
        pl.col('hit_at_k').mean(),
        pl.col('precision_at_k').mean()
    ))
    display(df_k.group_by('class').agg(
        pl.col('hit_at_k').mean(),
        pl.col('precision_at_k').mean()
    ).sort(pl.col('hit_at_k'), descending=True))


Metrics @ 1


hit_at_k,precision_at_k
f64,f64
null,null


class,hit_at_k,precision_at_k
str,f64,f64



Metrics @ 5


hit_at_k,precision_at_k
f64,f64
null,null


class,hit_at_k,precision_at_k
str,f64,f64



Metrics @ 10


hit_at_k,precision_at_k
f64,f64
null,null


class,hit_at_k,precision_at_k
str,f64,f64


In [14]:
sample_part = 'val'
for k in [1, 5, 10]:
    print(f"\nMetrics @ {k}")
    df_k = df.filter(pl.col('sample_part') == sample_part).with_columns(hits_k = pl.col("hit_flg").list.slice(0, k))\
        .with_columns(
            pos = pl.int_ranges(1, pl.col("hits_k").list.len() + 1),
            hit_at_k = pl.col("hits_k").list.max(),
            precision_at_k = pl.col("hits_k").list.mean()
        )
    display(df_k.select(
        pl.col('hit_at_k').mean(),
        pl.col('precision_at_k').mean()
    ))
    display(df_k.group_by('class').agg(
        pl.col('hit_at_k').mean(),
        pl.col('precision_at_k').mean()
    ).sort(pl.col('hit_at_k'), descending=True))


Metrics @ 1


hit_at_k,precision_at_k
f64,f64
null,null


class,hit_at_k,precision_at_k
str,f64,f64



Metrics @ 5


hit_at_k,precision_at_k
f64,f64
null,null


class,hit_at_k,precision_at_k
str,f64,f64



Metrics @ 10


hit_at_k,precision_at_k
f64,f64
null,null


class,hit_at_k,precision_at_k
str,f64,f64


In [15]:
sample_part = 'test'
for k in [1, 5, 10]:
    print(f"\nMetrics @ {k}")
    df_k = df.filter(pl.col('sample_part') == sample_part).with_columns(hits_k = pl.col("hit_flg").list.slice(0, k))\
        .with_columns(
            pos = pl.int_ranges(1, pl.col("hits_k").list.len() + 1),
            hit_at_k = pl.col("hits_k").list.max(),
            precision_at_k = pl.col("hits_k").list.mean()
        )
    display(df_k.select(
        pl.col('hit_at_k').mean(),
        pl.col('precision_at_k').mean()
    ))
    display(df_k.group_by('class').agg(
        pl.col('hit_at_k').mean(),
        pl.col('precision_at_k').mean()
    ).sort(pl.col('hit_at_k'), descending=True))


Metrics @ 1


hit_at_k,precision_at_k
f64,f64
null,null


class,hit_at_k,precision_at_k
str,f64,f64



Metrics @ 5


hit_at_k,precision_at_k
f64,f64
null,null


class,hit_at_k,precision_at_k
str,f64,f64



Metrics @ 10


hit_at_k,precision_at_k
f64,f64
null,null


class,hit_at_k,precision_at_k
str,f64,f64
